In [18]:
import os
import requests
import numpy as np
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

In [ ]:
dr_masked = pd.read_pickle('data/dr/text/leakage/dr_masked.pkl')
text_cols = ['masked']
cols_to_keep = ['id'] + text_cols
dr_text = dr_masked[cols_to_keep].copy()

## Ollama Embedding

In [ ]:
import os
import requests
import numpy as np
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

def _process_row_embedding(idx, row, model, text_column="masked"):
    try:
        text_value = str(row[text_column]) if pd.notna(row[text_column]) else ""

        response = requests.post(
            "http://localhost:11434/api/embeddings",
            json={
                "model": model,
                "prompt": text_value
            },
            timeout=120
        )

        if response.status_code != 200:
            print(f"Ollama error for ID {row.get('id')}: {response.text}")
            return None

        result = response.json()
        embedding = result.get("embedding")

        if not embedding:
            # Empty embedding
            return None

        return {
            "index": idx,
            "id": row.get("id", None),
            "embedding": embedding
        }

    except Exception as e:
        print(f"Ollama call failed for ID {row.get('id')}: {e}")
        return None


def create_embedding_datasets(
    df,
    text_column="masked",
    model_names=None,
    workers=6,
    output_prefix="data/dr/text/embeddings/"
):
    """
    Generate embeddings for multiple Ollama models in parallel per row.
    Keeps all original rows; inserts NaNs for rows that failed.
    """

    if model_names is None:
        model_names = ["embeddinggemma:latest", "qwen3-embedding:latest", "all-minilm"]

    if not os.path.exists(output_prefix):
        os.makedirs(output_prefix, exist_ok=True)

    for model_name in model_names:
        print(f"\nProcessing model: {model_name}")

        results = []
        rows_list = list(df.iterrows())
        total = len(rows_list)
        batch_size = workers * 4

        with tqdm(total=total) as pbar:
            for batch_start in range(0, total, batch_size):
                batch = rows_list[batch_start: batch_start + batch_size]

                futures = {}
                with ThreadPoolExecutor(max_workers=workers) as executor:
                    for idx, row in batch:
                        future = executor.submit(
                            _process_row_embedding,
                            idx,
                            row,
                            model_name,
                            text_column
                        )
                        futures[future] = idx

                    for future in as_completed(futures):
                        result = future.result()
                        if result is not None:
                            results.append(result)
                        pbar.update(1)

        # Determine embedding dimension from first successful embedding
        if results:
            emb_dim = len(results[0]["embedding"])
        else:
            print(f"No embeddings generated for {model_name}")
            emb_dim = 4096  # fallback dimension

        # Create full-size DataFrame filled with NaN
        full_emb_df = pd.DataFrame(
            np.nan,
            index=df.index,
            columns=[f"{text_column}_emb_{i}" for i in range(emb_dim)]
        )

        # Fill in successful embeddings
        for r in results:
            full_emb_df.loc[r["index"]] = r["embedding"]

        # Insert ID column
        if "id" in df.columns:
            full_emb_df.insert(0, "id", df["id"].values)

        # Save
        filename = os.path.join(output_prefix, f"{model_name.replace(':', '_')}.pkl")
        full_emb_df.to_pickle(filename)
        print(f"Saved embeddings for {model_name} -> {filename}")

    print("All models processed.")

In [22]:
create_embedding_datasets(
    df=dr_text,
    text_column="masked",
    model_names=[
        "embeddinggemma:latest",
        "qwen3-embedding:latest",
        #"all-minilm" # context size too small
    ],
    workers=6,
    output_prefix="data/dr/text/embeddings/"
)


Processing model: embeddinggemma:latest


100%|██████████| 8789/8789 [06:53<00:00, 21.28it/s]


Saved embeddings for embeddinggemma:latest -> data/dr/text/embeddings/embeddinggemma_latest.pkl

Processing model: qwen3-embedding:latest


100%|██████████| 8789/8789 [3:21:17<00:00,  1.37s/it]  


Saved embeddings for qwen3-embedding:latest -> data/dr/text/embeddings/qwen3-embedding_latest.pkl
All models processed.


## Huggingface Embedding

In [1]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
import os
from dotenv import load_dotenv

In [2]:
# Load .env variables
load_dotenv()

# Get HF token from environment
hf_token = os.getenv("HF_TOKEN")

In [3]:
dr_masked = pd.read_pickle('data/dr/text/leakage/dr_masked.pkl')
text_cols = ['masked']
cols_to_keep = ['id'] + text_cols
dr_text = dr_masked[cols_to_keep].copy()

In [4]:
def create_embedding_dataset(df, text_columns, embedding_model_names, output_prefix):
    for model_name in embedding_model_names:
        print(f"Processing model: {model_name}")
        model = SentenceTransformer(model_name, use_auth_token=hf_token)
        
        df_copy = df.copy()
        emb_columns_all = []

        for col in text_columns:
            embeddings = [
                model.encode(t) if pd.notna(t) else None
                for t in df_copy[col].tolist()
            ]

            # Determine embedding dimension
            emb_dim = len(next(e for e in embeddings if e is not None))
            emb_matrix = np.array([e if e is not None else np.zeros(emb_dim) for e in embeddings])

            # Create DataFrame for embeddings
            emb_df = pd.DataFrame(
                emb_matrix, 
                columns=[f'{col}_emb_{i}' for i in range(emb_dim)]
            )
            emb_columns_all.extend(emb_df.columns.tolist())

            # Concatenate embeddings to dataframe
            df_copy = pd.concat([df_copy, emb_df], axis=1)

        id_column = 'id'   # or pass it as a parameter
        keep_cols = [id_column] + emb_columns_all
        df_final = df_copy[keep_cols].copy()

        # Save dataset
        filename = f"{output_prefix}{model_name.replace('/', '_')}.pkl"
        df_final.to_pickle(filename)
        print(f"Saved dataset: {filename}\n")

In [ ]:
embedding_models = [
    "intfloat/e5-base-v2",
    "all-MiniLM-L12-v2",
    "all-mpnet-base-v2",
    "paraphrase-mpnet-base-v2"
]

# Call function
create_embedding_dataset(dr_text, text_cols, embedding_models, output_prefix="data/dr/text/embeddings/")